# 1. Introduction and notebook objective

This notebook implements the daily inference workflow for the production-ready one-month stock-direction classifier. It is designed to load the approved HGB production artifact from step 5, validate the manifest contract, compute the same feature set used during training, and generate a readable decision template for the current symbol universe.

The notebook is intentionally limited to operational inference. It does not execute trades, does not optimize positions, and does not assign a validated confidence layer beyond the descriptive outputs described in the PRD.

In [1]:
from __future__ import annotations

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

ctx: dict = {}
run_notes: list[str] = []
run_warnings: list[str] = []
run_errors: list[str] = []

print("Notebook runtime initialized.")

Notebook runtime initialized.


## 1.1 Production model and run scope

Expected inputs for this section:
- the project root and notebook output directory
- the selected universe CSV and model manifest path
- the production model artifact and feature contract generated in step 5

This code cell should establish the notebook’s scope: the notebook reads the production manifest, defines the symbol universe to evaluate, and makes the output location explicit so the run is reproducible and auditable. It should also document the purpose of the run as a descriptive daily decision template rather than an execution workflow.

In [2]:
ctx["workflow_name"] = "1-Month Direction Daily Prediction Notebook"
ctx["step"] = "6_predict_data"
ctx["run_started_at_utc"] = datetime.now(timezone.utc).isoformat()
ctx["decision_template_only"] = True

print(f"Workflow: {ctx['workflow_name']}")
print(f"Run started (UTC): {ctx['run_started_at_utc']}")

Workflow: 1-Month Direction Daily Prediction Notebook
Run started (UTC): 2026-09-13T16:29:58.216505+00:00


## 1.2 Safety and no-trade disclaimer

This section defines the operational guardrail for the notebook. It should state clearly that the notebook is a decision template only, that no order generation is performed, and that the final outputs are intended for human review. The markdown description should also note that any optional research-mode comparison is not an approved confidence layer or a trading recommendation.

In [3]:
NO_TRADE_DISCLAIMER = (
    "This notebook provides model predictions as a decision template for human review only. "
    "It does not execute trades, generate orders, or provide automated portfolio actions."
)

RESEARCH_MODE_DISCLAIMER = (
    "Research-mode consensus is descriptive and hypothesis-oriented. "
    "It is not a validated confidence layer and is not approved for automated action."
)

ctx["no_trade_disclaimer"] = NO_TRADE_DISCLAIMER
ctx["research_mode_disclaimer"] = RESEARCH_MODE_DISCLAIMER

print(NO_TRADE_DISCLAIMER)
print(RESEARCH_MODE_DISCLAIMER)

This notebook provides model predictions as a decision template for human review only. It does not execute trades, generate orders, or provide automated portfolio actions.
Research-mode consensus is descriptive and hypothesis-oriented. It is not a validated confidence layer and is not approved for automated action.


# 2. Configuration and file paths

This section establishes the fixed configuration values for the notebook. The code cell should define explicit project-relative paths, output naming conventions, the symbol universe location, the production manifest location, the optional research mode toggle, and the deterministic settings used for reproducibility.

Expected inputs:
- project root for `1-month-direction-classifier`
- universe CSV and per-symbol archive location
- `production_model_manifest.json` path
- optional MLP research flag

Outputs:
- a single configuration block that later code cells can reuse
- explicit metadata for output filenames and data provenance

In [4]:
cwd = Path.cwd().resolve()
project_root = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / "5_tune_model" / "production_model_manifest.json").exists() and (candidate / "data").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate classifier project root containing 5_tune_model/production_model_manifest.json and data/."
    )

output_dir = project_root / "6_predict_data"
data_dir = project_root / "data"
run_date = datetime.now(timezone.utc).date().isoformat()

ctx.update(
    {
        "project_root": project_root,
        "output_dir": output_dir,
        "data_dir": data_dir,
        "run_date": run_date,
        "research_mode_enabled": True,
        "random_seed": 42,
    }
)

print(f"Project root: {project_root}")
print(f"Output directory: {output_dir}")
print(f"Run date: {run_date}")

Project root: /workspace/projects/1-month-direction-classifier
Output directory: /workspace/projects/1-month-direction-classifier/6_predict_data
Run date: 2026-09-13


## 2.1 Project root and notebook output directory

This code cell should set the project root relative to the notebook directory and define the output folder used for prediction CSV and JSON artifacts. It should also establish standard naming conventions such as the run date, prediction file prefix, and research output naming so the notebook produces readable and consistent files.

In [5]:
ctx["output_dir"].mkdir(parents=True, exist_ok=True)

ctx["primary_csv_name"] = f"prediction_{ctx['run_date']}.csv"
ctx["dual_csv_name"] = f"prediction_dual_{ctx['run_date']}.csv"
ctx["run_json_name"] = f"prediction_run_{ctx['run_date']}.json"

ctx["primary_csv_path"] = ctx["output_dir"] / ctx["primary_csv_name"]
ctx["dual_csv_path"] = ctx["output_dir"] / ctx["dual_csv_name"]
ctx["run_json_path"] = ctx["output_dir"] / ctx["run_json_name"]

print("Output filenames configured:")
print(f"- {ctx['primary_csv_name']}")
print(f"- {ctx['dual_csv_name']}")
print(f"- {ctx['run_json_name']}")

Output filenames configured:
- prediction_2026-09-13.csv
- prediction_dual_2026-09-13.csv
- prediction_run_2026-09-13.json


## 2.2 Model manifest and universe inputs

Expected inputs:
- the model manifest path from step 5
- the universe CSV under the project data directory
- any optional research model paths or flags

This code cell should define the manifest path, universe path, optional research model path, and any metadata needed to resolve the correct production and research artifacts. It should keep the data flow explicit and make it obvious which path is operationally active and which path is optional-only.

In [6]:
manifest_path = ctx["project_root"] / "5_tune_model" / "production_model_manifest.json"
mlp_model_path = ctx["project_root"] / "5_tune_model" / "mlp_classifier_baseline_tuned_model.pkl"
mlp_scaler_path = ctx["project_root"] / "5_tune_model" / "mlp_classifier_baseline_tuned_scaler.pkl"

universe_candidates = sorted(ctx["data_dir"].glob("*.csv"))
preferred_universe = ctx["data_dir"] / "DEMO_003.csv"
universe_csv_path = preferred_universe if preferred_universe.exists() else (universe_candidates[0] if universe_candidates else None)

if universe_csv_path is None:
    raise FileNotFoundError("No universe CSV files were found under the data directory.")

universe_name = universe_csv_path.stem
universe_symbol_dir = ctx["data_dir"] / universe_name

ctx.update(
    {
        "manifest_path": manifest_path,
        "universe_csv_path": universe_csv_path,
        "universe_name": universe_name,
        "universe_symbol_dir": universe_symbol_dir,
        "mlp_model_path": mlp_model_path,
        "mlp_scaler_path": mlp_scaler_path,
    }
)

print(f"Manifest path: {manifest_path}")
print(f"Universe CSV: {universe_csv_path}")
print(f"Symbol directory: {universe_symbol_dir}")

Manifest path: /workspace/projects/1-month-direction-classifier/5_tune_model/production_model_manifest.json
Universe CSV: /workspace/projects/1-month-direction-classifier/data/DEMO_003.csv
Symbol directory: /workspace/projects/1-month-direction-classifier/data/DEMO_003


## 2.3 Runtime checks and deterministic configuration

This code cell should capture the required runtime libraries, random seed, and any deterministic execution settings relevant to scikit-learn and notebook execution. It should verify that the required environment dependencies are present or raise a clear user-facing warning if a crucial library is missing. The output should be a validated configuration state ready for later inference steps.

In [7]:
import importlib

required_libraries = [
    "pandas",
    "numpy",
    "sklearn",
    "joblib",
    "matplotlib",
    "seaborn",
]

missing_libraries = [lib for lib in required_libraries if importlib.util.find_spec(lib) is None]

np.random.seed(ctx["random_seed"])
random.seed(ctx["random_seed"])

ctx["required_libraries"] = required_libraries
ctx["missing_libraries"] = missing_libraries

if missing_libraries:
    run_warnings.append(f"Missing libraries detected: {missing_libraries}")

print(f"Random seed set to: {ctx['random_seed']}")
print(f"Missing libraries: {missing_libraries if missing_libraries else 'None'}")

Random seed set to: 42
Missing libraries: None


# 3. Production model manifest validation

This section validates the production artifact contract defined in step 5 before the notebook performs any inference. The code cells in this section should fail early or emit a clear warning if the manifest is missing, incomplete, or inconsistent with governance requirements. This prevents stale or malformed artifacts from being used silently in production.

In [8]:
manifest_validation = {
    "exists": False,
    "required_keys_present": False,
    "missing_keys": [],
    "json_loaded": False,
    "governance_passed": False,
    "model_file_exists": False,
    "errors": [],
    "warnings": [],
}

required_manifest_keys = [
    "model_name",
    "model_type",
    "model_path",
    "feature_columns",
    "label_mapping",
    "label_thresholds",
    "prediction_horizon_days",
    "scaler_required",
    "scaler_path",
    "training_data_end",
    "governance_status",
]

manifest: dict = {}
ctx["manifest_validation"] = manifest_validation
ctx["required_manifest_keys"] = required_manifest_keys
print("Manifest validation state initialized.")

Manifest validation state initialized.


## 3.1 Load manifest and confirm required keys

Expected inputs:
- `production_model_manifest.json` path
- JSON file generated in step 5

This code cell should load the manifest and verify that all required schema keys are present, including `model_name`, `model_type`, `model_path`, `feature_columns`, `label_mapping`, `label_thresholds`, `prediction_horizon_days`, `scaler_required`, `scaler_path`, `training_data_end`, and `governance_status`. It should also check for malformed JSON or missing nested content and set a validation flag that later cells can respect.

In [9]:
if ctx["manifest_path"].exists():
    manifest_validation["exists"] = True
    try:
        with open(ctx["manifest_path"], "r", encoding="utf-8") as f:
            manifest = json.load(f)
        manifest_validation["json_loaded"] = True
    except Exception as exc:
        manifest_validation["errors"].append(f"Failed to parse manifest JSON: {exc}")
else:
    manifest_validation["errors"].append(f"Manifest file not found: {ctx['manifest_path']}")

if manifest_validation["json_loaded"]:
    missing_keys = [k for k in required_manifest_keys if k not in manifest]
    manifest_validation["missing_keys"] = missing_keys
    manifest_validation["required_keys_present"] = len(missing_keys) == 0

    if missing_keys:
        manifest_validation["errors"].append(f"Missing required manifest keys: {missing_keys}")

ctx["manifest"] = manifest
print("Manifest loaded:", manifest_validation["json_loaded"])
print("Required keys present:", manifest_validation["required_keys_present"])

Manifest loaded: True
Required keys present: True


## 3.2 Validate model file and governance status

This code cell should confirm that the manifest’s `model_path` resolves to a real file in the project and that the selected production model is a legitimate inference artifact. It should then verify the governance check `governance_status.passed == true` before allowing the model path to be used for production prediction. If the governance requirement fails, the code should raise a clear error or halt the production path with an audit-friendly message.

In [10]:
if manifest_validation["required_keys_present"]:
    manifest_model_path = Path(manifest["model_path"])

    if manifest_model_path.is_absolute():
        candidate_paths = [manifest_model_path]
    else:
        candidate_paths = [
            ctx["project_root"] / manifest_model_path,
            ctx["project_root"].parent / manifest_model_path,
            ctx["project_root"] / "5_tune_model" / manifest_model_path.name,
        ]

    model_path = next((p for p in candidate_paths if p.exists()), candidate_paths[0])
    manifest_validation["model_file_exists"] = model_path.exists()
    ctx["production_model_path"] = model_path

    governance_status = manifest.get("governance_status", {})
    manifest_validation["governance_passed"] = bool(governance_status.get("passed", False))

    if not manifest_validation["model_file_exists"]:
        manifest_validation["errors"].append(f"Production model file missing: {model_path}")
    if not manifest_validation["governance_passed"]:
        manifest_validation["errors"].append("governance_status.passed is not true; production inference blocked.")
else:
    ctx["production_model_path"] = None

print("Resolved production model path:", ctx["production_model_path"])
print("Model file exists:", manifest_validation["model_file_exists"])
print("Governance passed:", manifest_validation["governance_passed"])

Resolved production model path: /workspace/projects/1-month-direction-classifier/5_tune_model/mlp_classifier_baseline_tuned_model.pkl
Model file exists: True
Governance passed: False


## 3.3 Record manifest warnings and stop conditions

Expected outputs:
- validation status object
- warnings or error reasons for incomplete schema
- a clean pass/fail state for downstream execution

This code cell should centralize any manifest validation results, including missing fields, missing files, or governance failures. It should produce a summary that can be printed to the notebook or stored in the final JSON metadata, while ensuring the production inference path cannot proceed when the manifest is invalid.

In [11]:
can_run_production = (
    manifest_validation["exists"]
    and manifest_validation["json_loaded"]
    and manifest_validation["required_keys_present"]
    and manifest_validation["model_file_exists"]
    and manifest_validation["governance_passed"]
)

ctx["can_run_production"] = can_run_production
ctx["manifest_validation"] = manifest_validation

run_warnings.extend(manifest_validation["warnings"])
run_errors.extend(manifest_validation["errors"])

print("Manifest validation summary:")
print(json.dumps(manifest_validation, indent=2))
print("Production inference enabled:", can_run_production)

Manifest validation summary:
{
  "exists": true,
  "required_keys_present": true,
  "missing_keys": [],
  "json_loaded": true,
  "governance_passed": false,
  "model_file_exists": true,
  "errors": [
    "governance_status.passed is not true; production inference blocked."
  ],
  "warnings": []
}
Production inference enabled: False


# 4. Universe and symbol data loading

This section resolves the active symbol universe and loads the local archive files for each symbol. The code cells here ensure the notebook uses only valid and sufficiently deep symbol histories, while logging skipped symbols and invalid schema cases so the human reviewer can understand why some names did not generate predictions.

In [12]:
symbol_histories: dict[str, pd.DataFrame] = {}
valid_histories: dict[str, pd.DataFrame] = {}
skipped_symbols: list[dict] = []

required_raw_columns = ["date", "open", "high", "low", "close", "adj_close", "volume"]
column_aliases = {
    "date": "date",
    "open": "open",
    "high": "high",
    "low": "low",
    "close": "close",
    "adjusted_close": "adj_close",
    "adj_close": "adj_close",
    "volume": "volume",
}

ctx["required_raw_columns"] = required_raw_columns
ctx["column_aliases"] = column_aliases
print("Data-loading containers initialized.")

Data-loading containers initialized.


## 4.1 Load universe CSV and required fields

Expected inputs:
- the selected universe CSV file under `data/{NAME}.csv`
- a required `symbol` field

This code cell should load the universe file, validate that the required field(s) are present, and create a clean symbol list for inference. It should also reject or warn clearly if the universe file is missing or if the selected data source does not match the expected format.

In [13]:
universe_df = pd.read_csv(ctx["universe_csv_path"])
universe_df.columns = [str(c).strip() for c in universe_df.columns]

symbol_column = None
for candidate in ["symbol", "Symbol", "SYMBOL", "ticker_symbol"]:
    if candidate in universe_df.columns:
        symbol_column = candidate
        break

if symbol_column is None:
    raise ValueError(f"Universe CSV is missing a symbol field. Found columns: {list(universe_df.columns)}")

universe_df["symbol"] = universe_df[symbol_column].astype(str).str.strip()
universe_df = universe_df[universe_df["symbol"].ne("")].copy()
symbols = sorted(universe_df["symbol"].unique().tolist())

ctx["universe_df"] = universe_df
ctx["symbols"] = symbols

print(f"Universe loaded from: {ctx['universe_csv_path']}")
print(f"Symbols found: {len(symbols)}")
print(symbols[:10])

Universe loaded from: /workspace/projects/1-month-direction-classifier/data/DEMO_003.csv
Symbols found: 3
['AAPL.US', 'AMZN.US', 'TSLA.US']


## 4.2 Resolve symbol archive files and history windows

Expected inputs:
- the symbol list from the universe
- sanitized per-symbol archive files from `data/{NAME}/{SYMBOL}.csv`

This code cell should resolve each symbol to its local archive file, load the symbol history in chronological order, and keep only the sanitized data relevant to the model. It should preserve temporal structure and verify that the available history is enough to satisfy the longest feature lookback windows, especially `sma_200` and the 60-day/20-day features described in step 2.

In [14]:
for symbol in ctx["symbols"]:
    symbol_file = ctx["universe_symbol_dir"] / f"{symbol}.csv"

    if not symbol_file.exists():
        skipped_symbols.append({"symbol": symbol, "reason": "missing_file", "details": str(symbol_file)})
        continue

    try:
        df = pd.read_csv(symbol_file)
    except Exception as exc:
        skipped_symbols.append({"symbol": symbol, "reason": "invalid_schema", "details": f"read_error: {exc}"})
        continue

    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns=ctx["column_aliases"])

    missing = [c for c in ctx["required_raw_columns"] if c not in df.columns]
    if missing:
        skipped_symbols.append({"symbol": symbol, "reason": "invalid_schema", "details": f"missing_columns: {missing}"})
        continue

    df = df[ctx["required_raw_columns"]].copy()
    df["symbol"] = symbol
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    numeric_cols = ["open", "high", "low", "close", "adj_close", "volume"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["date", "open", "high", "low", "close", "volume"]).sort_values("date")
    df = df[df["close"] > 0].reset_index(drop=True)

    symbol_histories[symbol] = df

ctx["symbol_histories"] = symbol_histories
print(f"Loaded histories for {len(symbol_histories)} symbols.")
print(f"Skipped so far: {len(skipped_symbols)}")

Loaded histories for 3 symbols.
Skipped so far: 0


## 4.3 Enforce minimum history and skip diagnostics

Expected outputs:
- a valid-symbol list
- a skipped-symbol diagnostics log with reason codes
- excluded entries for `insufficient_history`, `missing_file`, and `invalid_schema`

This code cell should check the minimum 200 trading days requirement, exclude symbols that fail the history threshold, and produce a diagnostics summary that is easy to review in the notebook and later export to metadata. The logic should be explicit about what counts as a valid day and what counts as unusable data.

In [15]:
MIN_HISTORY_DAYS = 200

for symbol, df in symbol_histories.items():
    if len(df) < MIN_HISTORY_DAYS:
        skipped_symbols.append(
            {
                "symbol": symbol,
                "reason": "insufficient_history",
                "details": f"rows={len(df)} < {MIN_HISTORY_DAYS}",
            }
        )
        continue

    valid_histories[symbol] = df

skipped_df = pd.DataFrame(skipped_symbols)
ctx["min_history_days"] = MIN_HISTORY_DAYS
ctx["valid_histories"] = valid_histories
ctx["skipped_df"] = skipped_df

print(f"Valid symbols after history checks: {len(valid_histories)}")
print(f"Skipped symbols logged: {len(skipped_df)}")
if not skipped_df.empty:
    display(skipped_df.head(10))

Valid symbols after history checks: 3
Skipped symbols logged: 0


# 5. Feature engineering parity with step 2

This section rebuilds the exact feature-engineering pipeline used during the training data preparation step so the inference-time feature contract matches the production model. The goal is to preserve the same 39-feature schema and ordering that the manifest declares, while only taking the most recent valid row for each symbol.

In [16]:
def compute_rsi_wilder(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["ret_1d"] = out["close"].pct_change(1)
    out["log_ret_1d"] = np.log(out["close"] / out["close"].shift(1))
    out["ret_5d"] = out["close"].pct_change(5)
    out["ret_10d"] = out["close"].pct_change(10)
    out["ret_20d"] = out["close"].pct_change(20)
    out["ret_60d"] = out["close"].pct_change(60)
    out["log_ret_5d"] = np.log(out["close"] / out["close"].shift(5))
    out["log_ret_20d"] = np.log(out["close"] / out["close"].shift(20))

    out["tr_range"] = out["high"] - out["low"]
    prev_close = out["close"].shift(1)
    out["true_range"] = np.maximum.reduce(
        [
            (out["high"] - out["low"]).values,
            (out["high"] - prev_close).abs().values,
            (out["low"] - prev_close).abs().values,
        ]
    )

    out["sma_5"] = out["close"].rolling(5, min_periods=5).mean()
    out["sma_20"] = out["close"].rolling(20, min_periods=20).mean()
    out["sma_50"] = out["close"].rolling(50, min_periods=50).mean()
    out["sma_200"] = out["close"].rolling(200, min_periods=200).mean()

    out["close_over_sma_20"] = out["close"] / out["sma_20"]
    out["close_over_sma_50"] = out["close"] / out["sma_50"]
    out["close_over_sma_200"] = out["close"] / out["sma_200"]
    out["sma_20_slope_5d"] = out["sma_20"] / out["sma_20"].shift(5) - 1
    out["sma_50_slope_5d"] = out["sma_50"] / out["sma_50"].shift(5) - 1

    out["vol_ret_5d"] = out["ret_1d"].rolling(5, min_periods=5).std()
    out["vol_ret_20d"] = out["ret_1d"].rolling(20, min_periods=20).std()
    out["vol_ret_60d"] = out["ret_1d"].rolling(60, min_periods=60).std()
    out["vol_tr_5d"] = out["true_range"].rolling(5, min_periods=5).std()
    out["vol_tr_20d"] = out["true_range"].rolling(20, min_periods=20).std()

    out["value_traded"] = out["close"] * out["volume"]
    out["vol_ma_20"] = out["volume"].rolling(20, min_periods=20).mean()
    out["vol_ma_60"] = out["volume"].rolling(60, min_periods=60).mean()
    out["vol_rel_20"] = out["volume"] / out["vol_ma_20"]

    out["value_traded_ma_20"] = out["value_traded"].rolling(20, min_periods=20).mean()
    out["value_traded_rel_20"] = out["value_traded"] / out["value_traded_ma_20"]

    out["rsi_14"] = compute_rsi_wilder(out["close"], period=14)

    ema12 = out["close"].ewm(span=12, adjust=False, min_periods=12).mean()
    ema26 = out["close"].ewm(span=26, adjust=False, min_periods=26).mean()
    out["macd"] = ema12 - ema26
    out["macd_signal"] = out["macd"].ewm(span=9, adjust=False, min_periods=9).mean()
    out["macd_hist"] = out["macd"] - out["macd_signal"]

    out["atr_14"] = out["true_range"].ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()

    out["hh_20d"] = out["high"].rolling(20, min_periods=20).max()
    out["ll_20d"] = out["low"].rolling(20, min_periods=20).min()
    out["dist_from_20d_high"] = out["close"] / out["hh_20d"] - 1
    out["dist_from_20d_low"] = out["close"] / out["ll_20d"] - 1

    out["fwd_ret_20d"] = out["close"].shift(-20) / out["close"] - 1

    return out


ctx["compute_features"] = compute_features
print("Feature engineering functions ready.")

Feature engineering functions ready.


## 5.1 Rebuild the same 39-feature contract

Expected inputs:
- sanitized symbol histories
- the feature specification created in step 2

This code cell should replicate the exact temporal feature logic used in the prepared data stage without changing the feature definitions. It should compute the same rolling, return, and technical indicators that the model was trained on and keep them aligned with the names and order declared in the production manifest.

In [17]:
engineered_by_symbol: dict[str, pd.DataFrame] = {}

for symbol, hist_df in valid_histories.items():
    try:
        engineered_by_symbol[symbol] = compute_features(hist_df)
    except Exception as exc:
        skipped_symbols.append({"symbol": symbol, "reason": "feature_error", "details": str(exc)})

ctx["engineered_by_symbol"] = engineered_by_symbol
ctx["skipped_df"] = pd.DataFrame(skipped_symbols)

print(f"Engineered features for symbols: {len(engineered_by_symbol)}")
print(f"Total skipped symbols: {len(ctx['skipped_df'])}")

Engineered features for symbols: 3
Total skipped symbols: 0


## 5.2 Keep only the most recent row per symbol

This code cell should reduce each valid symbol history to the latest available observation after feature engineering. It should preserve the correct date for the final row, keep all required market values such as close and volume, and confirm that the row is aligned with the same feature windows used in training. It should also note when the latest row is missing or partial so the later prediction step can skip that symbol cleanly.

In [18]:
if ctx["can_run_production"]:
    target_feature_columns = list(manifest["feature_columns"])
else:
    target_feature_columns = []

latest_rows = []
for symbol, feat_df in engineered_by_symbol.items():
    if not target_feature_columns:
        continue

    available = feat_df.dropna(subset=target_feature_columns)
    if available.empty:
        skipped_symbols.append(
            {"symbol": symbol, "reason": "latest_row_missing", "details": "No fully populated feature row."}
        )
        continue

    latest = available.iloc[-1].copy()
    latest_rows.append(latest)

latest_rows_df = pd.DataFrame(latest_rows)
ctx["target_feature_columns"] = target_feature_columns
ctx["latest_rows_df"] = latest_rows_df
ctx["skipped_df"] = pd.DataFrame(skipped_symbols)

print(f"Latest valid feature rows: {len(latest_rows_df)}")

Latest valid feature rows: 0


## 5.3 Verify feature column order against the manifest

Expected inputs:
- engineered feature matrix
- manifest `feature_columns` sequence

This code cell should compare the generated feature names and order to the exact sequence recorded in the manifest. It should raise a clear error if the order deviates, because a misordered feature set would invalidate the inference contract and compromise the model’s intended behavior. The output should be a validated feature dataset ready for scoring.

In [19]:
missing_feature_columns = []
if ctx["can_run_production"]:
    missing_feature_columns = [c for c in ctx["target_feature_columns"] if c not in ctx["latest_rows_df"].columns]

if missing_feature_columns:
    ctx["can_run_production"] = False
    run_errors.append(f"Feature mismatch vs manifest. Missing columns: {missing_feature_columns}")

if ctx["latest_rows_df"].empty:
    ctx["can_run_production"] = False
    run_errors.append("No valid latest rows are available for inference.")

if ctx["can_run_production"]:
    inference_features_df = ctx["latest_rows_df"][ctx["target_feature_columns"]].copy()
else:
    inference_features_df = pd.DataFrame(columns=ctx.get("target_feature_columns", []))

ctx["inference_features_df"] = inference_features_df
ctx["missing_feature_columns"] = missing_feature_columns

print(f"Can run production after feature checks: {ctx['can_run_production']}")
print(f"Inference matrix shape: {inference_features_df.shape}")

Can run production after feature checks: False
Inference matrix shape: (0, 0)


# 6. Production inference

This section performs the actual model scoring for each valid symbol. It should load the production HGB model and use the exact feature columns and label mapping from the manifest to convert raw predictions into user-readable `buy`, `hold`, and `sell` labels.

In [20]:
production_model = None
class_index_to_label: dict[int, str] = {}
label_to_class_index: dict[str, int] = {}

ctx["production_model"] = production_model
ctx["class_index_to_label"] = class_index_to_label
ctx["label_to_class_index"] = label_to_class_index

print("Production inference containers initialized.")

Production inference containers initialized.


## 6.1 Load HGB model and label mapping

Expected inputs:
- validated manifest
- production model artifact path
- label mapping and thresholds from the manifest

This code cell should load the HGB model, confirm it supports `predict()` and `predict_proba()`, and attach the manifest-provided label mapping and model metadata to the inference context. It should also retain the model version and governance metadata for later output rows and JSON metadata.

In [21]:
if ctx["can_run_production"]:
    import joblib

    production_model = joblib.load(ctx["production_model_path"])
    has_predict = hasattr(production_model, "predict")
    has_predict_proba = hasattr(production_model, "predict_proba")

    if not (has_predict and has_predict_proba):
        ctx["can_run_production"] = False
        run_errors.append("Loaded model does not support both predict() and predict_proba().")
    else:
        ctx["production_model"] = production_model
        label_to_class_index = {k: int(v) for k, v in manifest["label_mapping"].items()}
        class_index_to_label = {v: k for k, v in label_to_class_index.items()}
        ctx["label_to_class_index"] = label_to_class_index
        ctx["class_index_to_label"] = class_index_to_label
else:
    run_warnings.append("Production model load skipped due to earlier validation errors.")

print("Model loaded:", ctx.get("production_model") is not None)
print("Production active:", ctx["can_run_production"])

Model loaded: False
Production active: False


## 6.2 Run predict() and predict_proba() for each valid symbol

This code cell should iterate through the valid symbols, score the most recent engineered row, and produce a prediction label and raw probability list for each symbol. The output should include the raw model output indices so they can be mapped to `buy`, `hold`, and `sell` labels using the manifest’s `label_mapping` field.

In [22]:
raw_predictions = np.array([])
raw_probabilities = np.empty((0, 3))

if ctx["can_run_production"] and ctx.get("production_model") is not None and not ctx["inference_features_df"].empty:
    raw_predictions = ctx["production_model"].predict(ctx["inference_features_df"])
    raw_probabilities = ctx["production_model"].predict_proba(ctx["inference_features_df"])

ctx["raw_predictions"] = raw_predictions
ctx["raw_probabilities"] = raw_probabilities

print(f"Predictions generated: {len(raw_predictions)}")
print(f"Probability matrix shape: {raw_probabilities.shape}")

Predictions generated: 0
Probability matrix shape: (0, 3)


## 6.3 Map model outputs and derive signal strength

Expected outputs:
- `prediction` as `buy`, `hold`, or `sell`
- `prob_buy`, `prob_hold`, `prob_sell`
- `signal_strength` as a descriptive heuristic only

This code cell maps the model’s numeric class indexes to the label names, records the probability values in a machine-readable structure, and calculates a rough `signal_strength` heuristic derived from the probability distribution. The notebook should clearly describe that this heuristic is not a calibrated confidence measure and is intended only for human review.

In [23]:
primary_predictions_df = pd.DataFrame()

if len(ctx.get("raw_predictions", [])) > 0:
    latest_rows_df = ctx["latest_rows_df"].reset_index(drop=True)

    model_classes = list(ctx["production_model"].classes_)

    def class_position_for_label(label: str) -> int:
        desired_index = ctx["label_to_class_index"].get(label)

        if label in model_classes:
            return model_classes.index(label)
        if desired_index in model_classes:
            return model_classes.index(desired_index)
        if str(label) in [str(c) for c in model_classes]:
            return [str(c) for c in model_classes].index(str(label))
        if desired_index is not None and str(desired_index) in [str(c) for c in model_classes]:
            return [str(c) for c in model_classes].index(str(desired_index))

        raise KeyError(f"Could not map label '{label}' using model classes {model_classes}")

    buy_pos = class_position_for_label("buy")
    hold_pos = class_position_for_label("hold")
    sell_pos = class_position_for_label("sell")

    prob_buy = [float(row[buy_pos]) for row in ctx["raw_probabilities"]]
    prob_hold = [float(row[hold_pos]) for row in ctx["raw_probabilities"]]
    prob_sell = [float(row[sell_pos]) for row in ctx["raw_probabilities"]]

    pred_lookup = ctx["class_index_to_label"]
    predictions_text = []
    for c in ctx["raw_predictions"]:
        if c in {"buy", "hold", "sell"}:
            predictions_text.append(str(c))
        else:
            predictions_text.append(pred_lookup.get(int(c), f"class_{c}"))

    probs_arr = np.column_stack([prob_buy, prob_hold, prob_sell])
    sorted_probs = np.sort(probs_arr, axis=1)
    top_prob = sorted_probs[:, -1]
    gap_top2 = sorted_probs[:, -1] - sorted_probs[:, -2]
    signal_strength = 0.7 * top_prob + 0.3 * gap_top2

    primary_predictions_df = pd.DataFrame(
        {
            "date": pd.to_datetime(latest_rows_df["date"]).dt.date.astype(str),
            "symbol": latest_rows_df["symbol"].astype(str),
            "prediction": predictions_text,
            "prob_buy": prob_buy,
            "prob_hold": prob_hold,
            "prob_sell": prob_sell,
            "signal_strength": signal_strength,
            "model_name": manifest.get("model_name"),
            "model_version": manifest.get("created_at"),
            "prediction_horizon_days": manifest.get("prediction_horizon_days"),
            "close": latest_rows_df["close"].astype(float),
            "volume": latest_rows_df["volume"].astype(float),
            "fwd_ret_20d": latest_rows_df.get("fwd_ret_20d", pd.Series([np.nan] * len(latest_rows_df))),
        }
    )

ctx["primary_predictions_df"] = primary_predictions_df
print(f"Primary prediction rows: {len(primary_predictions_df)}")
if not primary_predictions_df.empty:
    display(primary_predictions_df.head(10))

Primary prediction rows: 0


# 7. Prediction outputs and metadata

This section assembles the final prediction outputs and run metadata for the primary production flow. It creates a human-readable CSV with symbol-level predictions, probability columns, model metadata, and a plain-language disclaimer that makes clear the notebook is a decision template and not a trade execution system.

In [24]:
output_artifacts = {
    "primary_csv": str(ctx["primary_csv_path"]),
    "run_json": str(ctx["run_json_path"]),
    "dual_csv": str(ctx["dual_csv_path"]),
}

ctx["output_artifacts"] = output_artifacts
print("Output artifact targets:")
print(json.dumps(output_artifacts, indent=2))

Output artifact targets:
{
  "primary_csv": "/workspace/projects/1-month-direction-classifier/6_predict_data/prediction_2026-09-13.csv",
  "run_json": "/workspace/projects/1-month-direction-classifier/6_predict_data/prediction_run_2026-09-13.json",
  "dual_csv": "/workspace/projects/1-month-direction-classifier/6_predict_data/prediction_dual_2026-09-13.csv"
}


## 7.1 Assemble the primary output CSV

Expected fields:
- `date`
- `symbol`
- `prediction`
- `prob_buy`
- `prob_hold`
- `prob_sell`
- `signal_strength`
- `model_name`
- `model_version`
- `prediction_horizon_days`
- `close`
- `volume`
- `fwd_ret_20d` where available, otherwise null/empty

This code cell should combine the symbol-level prediction results into a single primary output table, ensure the column names and order match the PRD, and save the CSV in the output directory using the required `prediction_yyyy-mm-dd.csv` naming convention.

In [25]:
required_primary_columns = [
    "date",
    "symbol",
    "prediction",
    "prob_buy",
    "prob_hold",
    "prob_sell",
    "signal_strength",
    "model_name",
    "model_version",
    "prediction_horizon_days",
    "close",
    "volume",
    "fwd_ret_20d",
]

primary_predictions_df = ctx.get("primary_predictions_df", pd.DataFrame())

if not primary_predictions_df.empty:
    primary_output_df = primary_predictions_df.copy()
    for col in required_primary_columns:
        if col not in primary_output_df.columns:
            primary_output_df[col] = np.nan

    primary_output_df = primary_output_df[required_primary_columns]
    primary_output_df.to_csv(ctx["primary_csv_path"], index=False)
    ctx["primary_output_df"] = primary_output_df
    print(f"Saved primary CSV: {ctx['primary_csv_path']}")
else:
    ctx["primary_output_df"] = pd.DataFrame(columns=required_primary_columns)
    run_warnings.append("Primary prediction output is empty; CSV not written.")
    print("No primary predictions available to save.")

No primary predictions available to save.


## 7.2 Save JSON run artifact and final summary

Expected outputs:
- `prediction_run_yyyy-mm-dd.json`
- generated_at timestamp
- model_name and model_version
- symbol_count and prediction_distribution
- manifest_reference and run_notes

This code cell should write the JSON metadata file and optionally print a concise notebook summary of the run: total symbol count, prediction distribution, and whether any symbols were skipped. It should include enough audit information for a reviewer to understand when and with which model artifact the inference was produced.

In [26]:
primary_predictions_df = ctx.get("primary_predictions_df", pd.DataFrame())
prediction_distribution = {}
if not primary_predictions_df.empty:
    prediction_distribution = primary_predictions_df["prediction"].value_counts(dropna=False).sort_index().to_dict()

run_metadata = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model_name": manifest.get("model_name") if manifest else None,
    "model_version": manifest.get("created_at") if manifest else None,
    "symbol_count": int(len(primary_predictions_df)),
    "prediction_distribution": prediction_distribution,
    "manifest_reference": str(ctx["manifest_path"]),
    "run_notes": run_notes,
    "warnings": run_warnings,
    "errors": run_errors,
    "no_trade_disclaimer": ctx["no_trade_disclaimer"],
}

with open(ctx["run_json_path"], "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)

ctx["run_metadata"] = run_metadata
print(f"Saved run metadata JSON: {ctx['run_json_path']}")
print(json.dumps({k: run_metadata[k] for k in ['model_name', 'symbol_count', 'prediction_distribution']}, indent=2))

Saved run metadata JSON: /workspace/projects/1-month-direction-classifier/6_predict_data/prediction_run_2026-09-13.json
{
  "model_name": "mlp_classifier_baseline",
  "symbol_count": 0,
  "prediction_distribution": {}
}


## 7.3 Capture disclaimers and skipped-symbol log

This code cell should append the operational warning that predictions are a decision template only and not a trade signal or execution instruction. It should also build a notebook-visible record of skipped symbols and their reasons, such as insufficient data, missing file, or invalid schema, so the run is transparent and governable.

In [27]:
if not ctx.get("skipped_df", pd.DataFrame()).empty:
    skipped_log_path = ctx["output_dir"] / f"skipped_symbols_{ctx['run_date']}.csv"
    ctx["skipped_df"].to_csv(skipped_log_path, index=False)
    ctx["skipped_log_path"] = skipped_log_path
    print(f"Saved skipped symbol diagnostics: {skipped_log_path}")
else:
    ctx["skipped_log_path"] = None
    print("No skipped symbols to log.")

print("Operational disclaimer:")
print(ctx["no_trade_disclaimer"])

No skipped symbols to log.
Operational disclaimer:
This notebook provides model predictions as a decision template for human review only. It does not execute trades, generate orders, or provide automated portfolio actions.


# 8. Optional research mode (MLP)

This section describes the optional MLP comparison path. The code in this section is research-only, explicitly non-production, and intended to give a descriptive comparison between the HGB production model and the MLP secondary model without assigning any validated confidence or execution authority.

In [28]:
research_context = {
    "enabled": bool(ctx.get("research_mode_enabled", False)),
    "model_loaded": False,
    "scaler_loaded": False,
}

research_model = None
research_scaler = None
research_predictions_df = pd.DataFrame()

ctx["research_context"] = research_context
ctx["research_model"] = research_model
ctx["research_scaler"] = research_scaler
ctx["research_predictions_df"] = research_predictions_df

print(f"Research mode enabled: {research_context['enabled']}")

Research mode enabled: True


## 8.1 Load research-only model and scaler

Expected inputs:
- the optional MLP artifact from step 5
- the StandardScaler required for the MLP pipeline
- the research-mode toggle

This code cell should load the secondary MLP model and its scaler only when the research flag is enabled. It should explicitly label the model as non-production and maintain a clear distinction between the production path and the optional comparison path.

In [29]:
if research_context["enabled"]:
    import joblib

    if ctx["mlp_model_path"].exists():
        research_model = joblib.load(ctx["mlp_model_path"])
        research_context["model_loaded"] = True
    else:
        run_warnings.append(f"Research model not found: {ctx['mlp_model_path']}")

    if ctx["mlp_scaler_path"].exists():
        research_scaler = joblib.load(ctx["mlp_scaler_path"])
        research_context["scaler_loaded"] = True
    else:
        run_warnings.append(
            f"Research scaler not found (optional if model does not need it): {ctx['mlp_scaler_path']}"
        )

ctx["research_model"] = research_model
ctx["research_scaler"] = research_scaler
ctx["research_context"] = research_context

print("Research model loaded:", research_context["model_loaded"])
print("Research scaler loaded:", research_context["scaler_loaded"])

Research model loaded: True
Research scaler loaded: False


## 8.2 Generate paired HGB/MLP predictions

This code cell should score the same valid symbols with the HGB and MLP models side by side and produce a comparison table with both predictions and probability vectors. The goal is descriptive comparison only, and the notebook should keep the comparison separate from the primary production output that is used for operational review.

In [30]:
if (
    research_context["enabled"]
    and research_context["model_loaded"]
    and not ctx["inference_features_df"].empty
    and not ctx["primary_predictions_df"].empty
):
    X_research = ctx["inference_features_df"].copy()

    if research_context["scaler_loaded"] and ctx["research_scaler"] is not None:
        X_research_arr = ctx["research_scaler"].transform(X_research)
    else:
        X_research_arr = X_research

    mlp_raw_pred = ctx["research_model"].predict(X_research_arr)
    mlp_raw_proba = ctx["research_model"].predict_proba(X_research_arr)

    class_order = [ctx["label_to_class_index"][label] for label in ["buy", "hold", "sell"]]
    class_position = {cls: idx for idx, cls in enumerate(ctx["research_model"].classes_)}

    mlp_prob_buy = [float(row[class_position[class_order[0]]]) for row in mlp_raw_proba]
    mlp_prob_hold = [float(row[class_position[class_order[1]]]) for row in mlp_raw_proba]
    mlp_prob_sell = [float(row[class_position[class_order[2]]]) for row in mlp_raw_proba]
    mlp_prediction = [ctx["class_index_to_label"].get(int(c), f"class_{c}") for c in mlp_raw_pred]

    research_predictions_df = ctx["primary_predictions_df"].copy()
    research_predictions_df["mlp_prediction"] = mlp_prediction
    research_predictions_df["mlp_prob_buy"] = mlp_prob_buy
    research_predictions_df["mlp_prob_hold"] = mlp_prob_hold
    research_predictions_df["mlp_prob_sell"] = mlp_prob_sell
else:
    research_predictions_df = pd.DataFrame()

ctx["research_predictions_df"] = research_predictions_df
print(f"Research paired predictions: {len(research_predictions_df)}")

Research paired predictions: 0


## 8.3 Build consensus buckets and descriptive status labels

Expected outputs:
- `consensus_bucket` values such as `hgb_buy_mlp_buy` and similar
- `consensus_status` values: `agreement`, `partial_disagreement`, or `model_conflict`

This code cell should create the descriptive 3x3 matrix of HGB and MLP label combinations and assign the consensus status using the PRD rules. It should use `model_conflict` only for explicit buy/sell contradictions such as `hgb_buy_mlp_sell` and `hgb_sell_mlp_buy`, while keeping all labels non-evaluative and non-actionable.

In [31]:
if not ctx["research_predictions_df"].empty:
    hgb_label = ctx["research_predictions_df"]["prediction"].astype(str)
    mlp_label = ctx["research_predictions_df"]["mlp_prediction"].astype(str)

    consensus_bucket = "hgb_" + hgb_label + "_mlp_" + mlp_label

    def consensus_status_fn(hgb: str, mlp: str) -> str:
        if hgb == mlp:
            return "agreement"
        if {hgb, mlp} == {"buy", "sell"}:
            return "model_conflict"
        return "partial_disagreement"

    consensus_status = [consensus_status_fn(h, m) for h, m in zip(hgb_label, mlp_label)]

    ctx["research_predictions_df"] = ctx["research_predictions_df"].assign(
        consensus_bucket=consensus_bucket,
        consensus_status=consensus_status,
    )

print("Consensus bucket/status columns prepared for research mode.")
if not ctx["research_predictions_df"].empty:
    display(
        ctx["research_predictions_df"][["symbol", "prediction", "mlp_prediction", "consensus_bucket", "consensus_status"]].head(10)
    )

Consensus bucket/status columns prepared for research mode.


# 9. Research-mode output artifacts

This section saves the optional dual-model output and adds a disclaimer that the consensus view is a hypothesis and has not been validated out-of-sample. The output should remain clearly secondary to the production model and should not imply a validated confidence layer or strategy decision.

In [32]:
ctx["dual_output_df"] = pd.DataFrame()
print("Research output containers initialized.")

Research output containers initialized.


## 9.1 Save the dual-model CSV and metadata

Expected outputs:
- `prediction_dual_yyyy-mm-dd.csv`
- all primary-mode columns plus `mlp_prediction`, `mlp_prob_buy`, `mlp_prob_hold`, `mlp_prob_sell`, `consensus_bucket`, and `consensus_status`

This code cell should write the research-only CSV alongside the primary output, ensuring the schema matches the PRD and the file remains distinct from the production prediction artifact. It should confirm that the dual-model file is explicitly marked as research-only in metadata and notebook display.

In [33]:
if not ctx["research_predictions_df"].empty:
    dual_columns = [
        "date",
        "symbol",
        "prediction",
        "prob_buy",
        "prob_hold",
        "prob_sell",
        "signal_strength",
        "model_name",
        "model_version",
        "prediction_horizon_days",
        "close",
        "volume",
        "fwd_ret_20d",
        "mlp_prediction",
        "mlp_prob_buy",
        "mlp_prob_hold",
        "mlp_prob_sell",
        "consensus_bucket",
        "consensus_status",
    ]

    dual_output_df = ctx["research_predictions_df"].copy()
    for col in dual_columns:
        if col not in dual_output_df.columns:
            dual_output_df[col] = np.nan

    dual_output_df = dual_output_df[dual_columns]
    dual_output_df.to_csv(ctx["dual_csv_path"], index=False)
    ctx["dual_output_df"] = dual_output_df
    print(f"Saved dual-model CSV: {ctx['dual_csv_path']}")
else:
    run_warnings.append("Research mode output is empty; dual CSV not written.")
    print("No research predictions available to save.")

No research predictions available to save.


## 9.2 Add explicit non-validated consensus disclaimer

This code cell should attach a plain-language disclaimer to the research CSV metadata and notebook output stating that consensus is descriptive and not validated out-of-sample. It should be clear that the dual-model comparison does not create automated decision authority, portfolio actions, or model confidence claims.

In [34]:
ctx["run_metadata"]["research_mode_enabled"] = bool(research_context["enabled"])
ctx["run_metadata"]["research_rows"] = int(len(ctx.get("dual_output_df", pd.DataFrame())))
ctx["run_metadata"]["research_mode_disclaimer"] = ctx["research_mode_disclaimer"]

if not ctx.get("dual_output_df", pd.DataFrame()).empty:
    ctx["run_metadata"]["consensus_status_distribution"] = (
        ctx["dual_output_df"]["consensus_status"].value_counts(dropna=False).sort_index().to_dict()
    )
else:
    ctx["run_metadata"]["consensus_status_distribution"] = {}

with open(ctx["run_json_path"], "w", encoding="utf-8") as f:
    json.dump(ctx["run_metadata"], f, indent=2)

print(ctx["research_mode_disclaimer"])

Research-mode consensus is descriptive and hypothesis-oriented. It is not a validated confidence layer and is not approved for automated action.


# 10. Final notebook review and future boundary

This final section acts as a quality gate for the notebook run. It verifies the required files exist, checks that the production path is the only validated decision path, and clearly documents the future step 6c boundary where walk-forward out-of-sample analysis, bucket-level return analysis, and calibration review will be handled separately.

In [35]:
final_checks = {
    "primary_csv_exists": ctx["primary_csv_path"].exists(),
    "run_json_exists": ctx["run_json_path"].exists(),
    "dual_csv_exists": ctx["dual_csv_path"].exists(),
    "skipped_log_exists": bool(ctx.get("skipped_log_path") and Path(ctx["skipped_log_path"]).exists()),
    "production_rows": int(len(ctx.get("primary_output_df", pd.DataFrame()))),
    "research_rows": int(len(ctx.get("dual_output_df", pd.DataFrame()))),
}

ctx["final_checks"] = final_checks
print(json.dumps(final_checks, indent=2))

if final_checks["production_rows"] == 0:
    run_warnings.append("Primary output contains zero prediction rows.")

{
  "primary_csv_exists": false,
  "run_json_exists": true,
  "dual_csv_exists": false,
  "skipped_log_exists": false,
  "production_rows": 0,
  "research_rows": 0
}


## 10.1 Verify outputs and file integrity

This code cell should check that the expected CSV and JSON artifacts were created, confirm the files are readable, and ensure the notebook has not silently skipped required outputs. It should also summarize the number of successful predictions and any skipped-symbol diagnostics for final review.

In [36]:
future_step_6c_boundary = {
    "step": "6c",
    "scope": [
        "Walk-forward out-of-sample evaluation",
        "Consensus bucket-level return and drawdown analysis",
        "Error-correlation analysis between HGB and MLP",
        "Calibration review before any confidence-layer promotion",
    ],
    "excluded_from_this_notebook": True,
}

ctx["future_step_6c_boundary"] = future_step_6c_boundary

run_notes.append("Step 6c evaluation is out of scope for this notebook and must remain a separate workflow.")
ctx["run_metadata"]["run_notes"] = run_notes
with open(ctx["run_json_path"], "w", encoding="utf-8") as f:
    json.dump(ctx["run_metadata"], f, indent=2)

print("Future boundary documented:")
print(json.dumps(future_step_6c_boundary, indent=2))
print("Final guardrail: this notebook is a decision template, not an execution engine.")

Future boundary documented:
{
  "step": "6c",
  "scope": [
    "Walk-forward out-of-sample evaluation",
    "Consensus bucket-level return and drawdown analysis",
    "Error-correlation analysis between HGB and MLP",
    "Calibration review before any confidence-layer promotion"
  ],
  "excluded_from_this_notebook": true
}
Final guardrail: this notebook is a decision template, not an execution engine.


## 10.2 Document the future step 6c boundary

This code cell should record the scope boundary for the current notebook and note that step 6c will cover walk-forward out-of-sample analysis, bucket-level return analysis, error correlation checks, and calibration review. It should explicitly avoid embedding any consensus confidence claims in the current production inference notebook and keep all future evaluation work clearly separated from the operational daily prediction workflow.

In [37]:
boundary_note = (
    "Open question noted from PRD: step 6c success criteria thresholds are not numerically specified "
    "for confidence-layer promotion. This notebook records outputs only; threshold definition "
    "must be finalized in the dedicated 6c evaluation notebook."
)

ctx["future_boundary_open_question"] = boundary_note
run_notes.append(boundary_note)
ctx["run_metadata"]["run_notes"] = run_notes

with open(ctx["run_json_path"], "w", encoding="utf-8") as f:
    json.dump(ctx["run_metadata"], f, indent=2)

print(boundary_note)

Open question noted from PRD: step 6c success criteria thresholds are not numerically specified for confidence-layer promotion. This notebook records outputs only; threshold definition must be finalized in the dedicated 6c evaluation notebook.
